In [1]:
import sys
import os
from collections import defaultdict
import pandas as pd
import scanpy as sc
import squidpy as sq
import numpy as np
import matplotlib.pyplot as plt
from glmpca import glmpca
from itertools import combinations
import torch

import sys
from importlib import reload

import gaston
from gaston import neural_net,cluster_plotting, dp_related, segmented_fit, restrict_spots, model_selection
from gaston import binning_and_plotting, isodepth_scaling, run_slurm_scripts, parse_adata
from gaston import spatial_gene_classification, plot_cell_types, filter_genes, process_NN_output

import seaborn as sns
import math

/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [ ]:
df = pd.read_parquet("./skin_A1_data/spatial/tissue_positions.parquet")
df.to_csv("./skin_A1_data/spatial/tissue_positions_list.csv", index=False)  
df

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_008um_00000_00000-1,0,0,0,22505.263073,25072.748986
1,s_008um_00000_00001-1,0,0,1,22505.476012,25056.786425
2,s_008um_00000_00002-1,0,0,2,22505.688951,25040.823864
3,s_008um_00000_00003-1,0,0,3,22505.901890,25024.861303
4,s_008um_00000_00004-1,0,0,4,22506.114830,25008.898743
...,...,...,...,...,...,...
702239,s_008um_00837_00833-1,0,837,833,36043.026674,11954.520977
702240,s_008um_00837_00834-1,0,837,834,36043.239499,11938.558619
702241,s_008um_00837_00835-1,0,837,835,36043.452324,11922.596261
702242,s_008um_00837_00836-1,0,837,836,36043.665149,11906.633904


In [2]:
data_folder='skin_A1_data' # folder with filtered_feature_bc_matrix.h5ad
spot_umi_threshold=50 # filter out cells with low UMIs

########################################################

adata=sq.read.visium(data_folder)
sc.pp.filter_cells(adata, min_counts=spot_umi_threshold)

gene_labels=adata.var.index.to_numpy()
counts_mat=np.array(adata.X.todense())
coords_mat=np.array(np.array(adata.obs[["array_row", "array_col"]]))

# make sure counts_mat is NxG and coords_mat is Nx2
if counts_mat.shape[0] != coords_mat.shape[0]:
    counts_mat=counts_mat.T

/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/squidpy/read/_read.py:82: DtypeWarning: Columns (1,2,3,4,5) have mixed types. Specify dtype option on import or set low_memory=False.
  coords = pd.read_csv(
/home/siyuan/miniconda3/envs/biovis/lib/python3.11/site-packages/anndata/_core/anndata.py:1793: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [ ]:
# save matrices
np.save('skin_A1_data/counts_mat.npy', counts_mat)
np.save('skin_A1_data/coords_mat.npy', coords_mat)
np.save('skin_A1_data/gene_labels.npy', gene_labels)

In [3]:
use_RGB=True # set to False if you do not want to use RGB as features

# CODE to compute R,G,B mean from RGB image
if use_RGB:
    rgb_mean=parse_adata.use_RGB(adata)

calculating RGB


  0%|          | 0/145474 [00:00<?, ?/s]

100%|██████████| 145474/145474 [02:50<00:00, 855.59/s]


In [4]:
num_dims=5
clip=0.01 # have to clip values to be very small!

A = parse_adata.get_top_pearson_residuals(num_dims,counts_mat,coords_mat,clip=clip)
if use_RGB:
    A=np.hstack((A,rgb_mean)) # attach to RGB mean
np.save('skin_A1_data/analytic_pearson.npy', A)

/home/siyuan/miniconda3/envs/biovis/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
/home/siyuan/miniconda3/envs/biovis/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


: 

In [ ]:
# visualize top GLM-PCs
rotated_coords=dp_related.rotate_by_theta(coords_mat, -np.pi/2)
R=2
C=4
fig,axs=plt.subplots(R,C,figsize=(20,10))
for r in range(R):
    for c in range(C):
        i=r*C+c
        axs[r,c].scatter(rotated_coords[:,0], rotated_coords[:,1], c=A[:,i],cmap='Reds',s=3)
        axs[r,c].set_title(f'PC{i}')